In [1]:
# import sys
#from podio import root_io
import uproot 
import awkward as ak
import math
import vector
import numpy as np
path = "/work/eic/users/aabhishe/EIC_3He_5x41_XRot_pi_bc.hepmc3.tree_sim_output_recon.root"

In [ ]:
tree = uproot.open(f"{path}:events")

# Read arrays
e_prime_pdg_arrays = tree["ReconstructedChargedParticles.PDG"].array()
e_beam_pdg_arrays = tree["MCParticles.PDG"].array()
mc_gen_status = tree["MCParticles.generatorStatus"].array()

#e_beam_energy = 5.0
#p_beam_energy = 41.0

e_prime_mask = (e_prime_pdg_arrays == 11) & (tree["ReconstructedChargedParticles.energy"].array() > 0.2)

e_beam_mask = (e_beam_pdg_arrays == 11) & (mc_gen_status == 4)
p_beam_mask = (e_beam_pdg_arrays == 1000020030 ) & (mc_gen_status == 4)
# Apply masks for scattered (reco) electrons
e_prime_pz = tree["ReconstructedChargedParticles.momentum.z"].array()[e_prime_mask]
e_prime_px = tree["ReconstructedChargedParticles.momentum.x"].array()[e_prime_mask]
e_prime_py = tree["ReconstructedChargedParticles.momentum.y"].array()[e_prime_mask]
e_prime_E = tree["ReconstructedChargedParticles.energy"].array()[e_prime_mask]


# Apply masks for beam (MC truth) electrons
e_beam_pz = tree["MCParticles.momentum.z"].array()[e_beam_mask]
e_beam_px = tree["MCParticles.momentum.x"].array()[e_beam_mask]
e_beam_py = tree["MCParticles.momentum.y"].array()[e_beam_mask]
# e_beam_E = ak.full_like(e_beam_pz, e_beam_energy)
e_beam_mass = 0.000511  # Electron mass in GeV
e_beam_E = np.sqrt(e_beam_px**2 + e_beam_py**2 + e_beam_pz**2 + e_beam_mass**2)

p_beam_pz = tree["MCParticles.momentum.z"].array()[p_beam_mask]
p_beam_px = tree["MCParticles.momentum.x"].array()[p_beam_mask]
p_beam_py = tree["MCParticles.momentum.y"].array()[p_beam_mask]
# p_beam_E = ak.full_like(p_beam_pz, p_beam_energy)
p_beam_mass = 2.8084  # Helium-3 mass in GeV
p_beam_E = np.sqrt(p_beam_px**2 + p_beam_py**2 + p_beam_pz**2 + p_beam_mass**2)



e_beam = vector.zip({
    "px": e_beam_px,
    "py": e_beam_py,
    "pz": e_beam_pz,
    "E": e_beam_E
})

e_prime = vector.zip({
    "px": e_prime_px,
    "py": e_prime_py,
    "pz": e_prime_pz,
    "E": e_prime_E
})

p_beam = vector.zip({
    "px": p_beam_px,
    "py": p_beam_py,
    "pz": p_beam_pz,
    "E": p_beam_E
})
sorted_indices = ak.argsort(e_prime.E, ascending=False)
e_prime_sorted = e_prime[sorted_indices]

print(f"e_beam: {e_beam[:10]}")
e_beam_single = ak.firsts(e_beam)
print('e_beam_single:', e_beam_single[:10])
p_beam_single = ak.firsts(p_beam)
e_prime_leading = ak.firsts(e_prime_sorted)

#four momentum transfer
q = e_beam_single - e_prime_leading
Q2 = -q.mass2

#bjorken x
x = 3.*Q2 / (2. * p_beam_single.dot(q))

print(f"Q2: {Q2[:10]}")
print(f"x: {x[:10]}")
Q2_clean = ak.to_numpy(ak.drop_none(Q2)).astype(np.float64)
x_clean = ak.to_numpy(ak.drop_none(x)).astype(np.float64)


e_beam: [[{x: 0.000592, y: -0.00112, z: -5, t: 5}], ..., [{x: -0.000111, y: ..., ...}]]
e_beam_single: [{x: 0.000592, y: -0.00112, z: -5, t: 5}, ..., {x: -0.000111, y: 0.000222, ...}]
Q2: [2.48, 2.27, 1.66, 1.43, 2.55, 1.06, 1.18, 1.52, 3.92, 20.6]
x: [0.0305, 0.0243, 0.0243, 0.0273, 0.00567, ..., 0.0127, 0.0166, 0.0111, 0.129]


In [5]:
import ROOT
Q2_bin_edges = np.logspace(-1, 3, 101)  # 100 bins from 10^-2 to 10^2

Q2_hist = ROOT.TH1F("his1", "Q2 Distribution; Q2 [GeV^2]; Events", len(Q2_bin_edges) - 1, Q2_bin_edges)
xbj_bin_edges = np.logspace(-4, 0, 101)  # 100 bins from 10^-4 to 10^-1
x_hist = ROOT.TH1F("his2", "Bjorken x Distribution; x; Events", len(xbj_bin_edges) - 1, xbj_bin_edges)

Q2_xbj_2d_hist = ROOT.TH2F("his2D", "Q2 vs Bjorken x; Q2 [GeV^2]; x", len(Q2_bin_edges) - 1, Q2_bin_edges, len(xbj_bin_edges) - 1, xbj_bin_edges)

Q2_hist.FillN(len(Q2_clean), Q2_clean, np.ones(len(Q2_clean), dtype=np.float64))
x_hist.FillN(len(x_clean), x_clean, np.ones(len(x_clean), dtype=np.float64))
Q2_xbj_2d_hist.FillN(len(Q2_clean), Q2_clean, x_clean, np.ones(len(Q2_clean), dtype=np.float64))
canvas= ROOT.TCanvas("canvas", "Q2 and x Distributions", 800, 600)
canvas1= ROOT.TCanvas("canvas1", "Q2 vs x Distribution", 800, 600)
canvas1.cd()
canvas1.SetLogy()
canvas1.SetLogx()
canvas1.SetLogz()
Q2_xbj_2d_hist.Draw("COLZ")
canvas1.SaveAs("/home/aabhishe/eic/EIC_Full_Sim/debug/Q2_x_2D_distribution_debug.pdf")
canvas.Divide(2, 1)
canvas.cd(1)
ROOT.gPad.SetLogy()
ROOT.gPad.SetLogx()
Q2_hist.Draw()
canvas.cd(2)
ROOT.gPad.SetLogx()
ROOT.gPad.SetLogy()
x_hist.Draw()



canvas.SaveAs("/home/aabhishe/eic/EIC_Full_Sim/debug/Q2_x_distributions_debug.pdf")


Warning in <TROOT::Append>: Replacing existing TH1: his1 (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: his2 (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: his2D (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas
Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas1
Info in <TCanvas::Print>: pdf file /home/aabhishe/eic/EIC_Full_Sim/debug/Q2_x_2D_distribution_debug.pdf has been created
Info in <TCanvas::Print>: pdf file /home/aabhishe/eic/EIC_Full_Sim/debug/Q2_x_distributions_debug.pdf has been created


In [ ]:
e_prime = vector.zip({
    "px": momentum_x_arrays,
    "py": momentum_y_arrays,
    "pz": momentum_z_arrays,
    "E": energy_arrays
})

In [ ]:
print(e_prime.pt[3])
print(e_prime.eta[3])